In [0]:
%run ../common/01_config

In [0]:
%run ../common/02_adls_connection

In [0]:
stream_df = (
    spark.read
    .option("multiline", "false")
    .json(bronze_stream_orders)
)


In [0]:
stream_df.printSchema()

In [0]:
from pyspark.sql.functions import col, trim, upper, to_timestamp, current_timestamp

stream_silver_df = (
    stream_df
    .select(
        col("order_id").cast("long"),
        col("amount").cast("double"),
        upper(trim(col("city"))).alias("city"),
        to_timestamp(col("timestamp")).alias("event_timestamp"),
        col("PartitionId"),
        col("EventEnqueuedUtcTime"),
        col("EventProcessedUtcTime")
    )
    .dropDuplicates(["order_id"])
    .filter(col("order_id").isNotNull())
    .filter(col("amount").isNotNull())
    .filter(col("amount") > 0)
    .withColumn("processed_at", current_timestamp())
)

In [0]:
display(stream_silver_df)

In [0]:
# write to silver

stream_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_stream_orders)